In [0]:
%run 
"./00setup_config"

Connected. Database ready.


path,name,size,modificationTime
abfss://landing@medallionhealthdata03.dfs.core.windows.net/appointments.csv,appointments.csv,10907,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/billing.csv,billing.csv,10018,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/doctors.csv,doctors.csv,962,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/patients.csv,patients.csv,5626,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/treatments.csv,treatments.csv,11072,1783938174000


Batch ID : f31778da-d3c9-44fd-b31e-89c6af23d7c0
Pipeline Version : 1.0
Run Time : 2026-07-13 11:07:53.607918


# Setting metadata

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


batch_id,source_name,layer,rows_read,rows_written,status,start_time,end_time,error_message


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Silver Tables

In [0]:
patients_df = spark.read.format("delta").load(f"{silver_path}patients")
appointments_df = spark.read.format("delta").load(f"{silver_path}appointments")
billing_df = spark.read.format("delta").load(f"{silver_path}billing")
doctors_df = spark.read.format("delta").load(f"{silver_path}doctors")
treatments_df = spark.read.format("delta").load(f"{silver_path}treatments")

In [0]:
patients_df = patients_df.withColumnRenamed("Patient_ID", "patient_id")

In [0]:
print("Patients :",patients_df.count())
print("Doctors :",doctors_df.count())
print("Appointments :",appointments_df.count())
print("Billing :",billing_df.count())
print("Treatments :",treatments_df.count())

Patients : 50
Doctors : 10
Appointments : 200
Billing : 200
Treatments : 200


# Gold Data Validation

In [0]:

patients_df = patients_df.dropDuplicates(["patient_id"])
appointments_df = appointments_df.dropDuplicates(["appointment_id"])
billing_df = billing_df.dropDuplicates(["bill_id"])
doctors_df = doctors_df.dropDuplicates(["doctor_id"])
treatments_df = treatments_df.dropDuplicates(["treatment_id"])

In [0]:
patients_df = patients_df.filter(F.col("patient_id").isNotNull())

appointments_df = appointments_df.filter(
    (F.col("appointment_id").isNotNull()) &
    (F.col("patient_id").isNotNull()) &
    (F.col("doctor_id").isNotNull())
)

billing_df = billing_df.filter(
    (F.col("bill_id").isNotNull()) &
    (F.col("patient_id").isNotNull())
)

doctors_df = doctors_df.filter(
    F.col("doctor_id").isNotNull()
)

treatments_df = treatments_df.filter(
    (F.col("treatment_id").isNotNull()) &
    (F.col("appointment_id").isNotNull())
)
     

In [0]:
billing_df = billing_df.filter(F.col("amount")>=0)

treatments_df = treatments_df.filter(F.col("cost")>=0)

In [0]:
patients_df = patients_df.withColumnRenamed("Patient_ID", "patient_id")

In [0]:

patient360 = (
patients_df.alias("p")
.join(
appointments_df.alias("a"),
F.col("p.patient_id")==F.col("a.patient_id"),
"left"
)
.join(
doctors_df.alias("d"),
F.col("a.doctor_id")==F.col("d.doctor_id"),
"left"
)
.join(
treatments_df.alias("t"),
F.col("a.appointment_id")==F.col("t.appointment_id"),
"left"
)
.join(
billing_df.alias("b"),
(
(F.col("p.patient_id")==F.col("b.patient_id"))
&
(F.col("t.treatment_id")==F.col("b.treatment_id"))
),
"left"
)
)

In [0]:

patient360 = patient360.filter(
    F.col("doctor_name").isNotNull()
)

patient360 = patient360.filter(
    F.col("hospital_branch").isNotNull()
)
     

In [0]:
patient360 = patient360.select(
F.col("p.patient_id"),
F.col("p.First_Name"),
F.col("p.Last_Name"),
F.col("p.Gender"),
F.col("p.date_of_birth"),
F.col("p.insurance_provider"),
F.col("d.doctor_name"),
F.col("d.specialization"),
F.col("d.hospital_branch"),
F.col("d.years_experience"),
F.col("a.appointment_id"),
F.col("a.appointment_date"),
F.col("a.status"),
F.col("a.reason_for_visit"),
F.col("t.treatment_id"),
F.col("t.treatment_type"),
F.col("t.cost"),
F.col("b.bill_id"),
F.col("b.amount"),
F.col("b.payment_method"),
F.col("b.payment_status")
)

In [0]:

patient360 = patient360.fillna({

"payment_method":"UNKNOWN",

"payment_status":"UNKNOWN",

"treatment_type":"UNKNOWN"

})
     

In [0]:
patient360 = patient360.filter(
    F.col("hospital_branch") != "UNKNOWN"
)

In [0]:
patient360.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}patient_360")

In [0]:
display(
spark.read
.format("delta")
.load(f"{gold_path}patient_360")
)

patient_id,First_Name,Last_Name,Gender,date_of_birth,insurance_provider,doctor_name,specialization,hospital_branch,years_experience,appointment_id,appointment_date,status,reason_for_visit,treatment_id,treatment_type,cost,bill_id,amount,payment_method,payment_status
P001,David,Williams,F,1955-06-04,WellnessCorp,Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,26,A197,2023-04-01,NO-SHOW,Emergency,T197,PHYSIOTHERAPY,975.49,B197,975.49,CASH,PENDING
P034,Alex,Smith,F,1950-01-26,WellnessCorp,Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,26,A043,2023-03-29,NO-SHOW,Consultation,T043,X-RAY,3207.25,B043,3207.25,INSURANCE,FAILED
P008,David,Davis,F,1976-07-05,WellnessCorp,Jane Davis,PEDIATRICS,EASTSIDE CLINIC,24,A194,2023-04-06,SCHEDULED,Therapy,T194,PHYSIOTHERAPY,1903.17,B194,1903.17,CASH,PENDING
P016,Michael,Taylor,M,2000-07-22,PulseSecure,Linda Brown,DERMATOLOGY,WESTSIDE CLINIC,5,A052,2023-07-12,NO-SHOW,Therapy,T052,ECG,2090.4,B052,2090.4,CASH,PAID
P019,Sarah,Miller,M,1975-05-24,WellnessCorp,Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,26,A193,2023-09-15,CANCELLED,Therapy,T193,PHYSIOTHERAPY,2446.24,B193,2446.24,CASH,FAILED
P037,Robert,Williams,M,1999-02-05,HealthIndia,Sarah Smith,PEDIATRICS,CENTRAL HOSPITAL,26,A017,2023-07-11,SCHEDULED,Emergency,T017,MRI,1655.49,B017,1655.49,CREDIT CARD,PENDING
P012,Laura,Davis,F,1991-12-08,MedCare Plus,Jane Davis,PEDIATRICS,EASTSIDE CLINIC,24,A174,2023-10-31,CANCELLED,Follow-up,T174,CHEMOTHERAPY,3384.37,B174,3384.37,CASH,PAID
P016,Michael,Taylor,M,2000-07-22,PulseSecure,Linda Wilson,ONCOLOGY,EASTSIDE CLINIC,21,A008,2023-05-24,CANCELLED,Consultation,T008,PHYSIOTHERAPY,3413.64,B008,3413.64,CASH,FAILED
P048,Emily,Miller,M,1983-03-24,PulseSecure,David Jones,PEDIATRICS,CENTRAL HOSPITAL,28,A003,2023-06-28,CANCELLED,Consultation,T003,MRI,3731.55,B003,3731.55,INSURANCE,PAID
P018,Laura,Wilson,M,1979-09-24,PulseSecure,Alex Davis,PEDIATRICS,CENTRAL HOSPITAL,23,A172,2023-03-09,SCHEDULED,Checkup,T172,X-RAY,2057.45,B172,2057.45,CASH,PAID


In [0]:

analytics_df = patient360.filter(

    (F.col("doctor_name").isNotNull()) &
    (F.col("hospital_branch").isNotNull()) &
    (F.col("appointment_date").isNotNull()) &
    (F.col("amount").isNotNull()) &
    (F.col("cost").isNotNull())

)
     

In [0]:
analytics_df.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}analytics_dataset")

# Revenue Analytics
## Revenue by doctor

In [0]:
revenue_by_doctor = (
    patient360
    .groupBy(
        "doctor_name",
        "specialization",
        "hospital_branch"
    )
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill"),
        F.countDistinct("patient_id").alias("total_patients"),
        F.countDistinct("appointment_id").alias("total_appointments"),
        F.countDistinct("treatment_id").alias("total_treatments")
    )
    .fillna(0)
)

In [0]:
doctor_window = Window.orderBy(F.desc("total_revenue"))

revenue_by_doctor = (
    revenue_by_doctor
    .withColumn(
        "doctor_rank",
        F.dense_rank().over(doctor_window)
    )
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
revenue_by_doctor.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}revenue_by_doctor")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


#Revenue by hospital

In [0]:
revenue_by_branch = (
    patient360
    .groupBy("hospital_branch")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.countDistinct("patient_id").alias("patients"),
        F.countDistinct("doctor_name").alias("doctors"),
        F.countDistinct("appointment_id").alias("appointments"),
        F.countDistinct("treatment_id").alias("treatments")
    )
    .fillna(0)
)

In [0]:
revenue_by_branch.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}revenue_by_branch")

# Monthly revenue

In [0]:
monthly_revenue = (
    analytics_df
    .filter(F.col("appointment_date").isNotNull())
    .withColumn("year", F.year("appointment_date"))
    .withColumn("month", F.month("appointment_date"))
    .groupBy("year", "month")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill"),
        F.count("bill_id").alias("total_bills")
    )
)

monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_path}monthly_revenue")
     

In [0]:
top_doctors = (
    revenue_by_doctor
    .orderBy(
        F.desc("total_revenue")
    )
    .limit(10)
)

In [0]:
top_doctors.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}top_doctors")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
top_patients = (
    patient360
    .groupBy(
        "patient_id",
        "First_Name",
        "Last_Name"
    )
    .agg(
        F.sum("amount").alias("total_spending"),
        F.countDistinct("appointment_id").alias("appointments")
    )
)

In [0]:

patient_window = Window.orderBy(
    F.desc("total_spending")
)

top_patients = (
    top_patients
    .withColumn(
        "patient_rank",
        F.dense_rank().over(patient_window)
    )
    .limit(10)
)
     

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
top_patients.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}top_patients")

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(revenue_by_doctor)

display(revenue_by_branch)

display(monthly_revenue)

display(top_doctors)

display(top_patients)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


doctor_name,specialization,hospital_branch,total_revenue,average_bill,total_patients,total_appointments,total_treatments,doctor_rank
Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,82696.47999999998,2851.602758620689,23,29,29,1
Alex Davis,PEDIATRICS,CENTRAL HOSPITAL,69586.1,2899.4208333333336,19,24,24,2
David Taylor,DERMATOLOGY,WESTSIDE CLINIC,66585.39,2663.4156,20,25,25,3
Jane Davis,PEDIATRICS,EASTSIDE CLINIC,59803.45999999999,2847.783809523809,17,21,21,4
Linda Brown,DERMATOLOGY,WESTSIDE CLINIC,53427.42,3339.21375,12,16,16,5
Jane Smith,PEDIATRICS,EASTSIDE CLINIC,52791.40999999999,2399.609545454545,18,22,22,6
Linda Wilson,ONCOLOGY,EASTSIDE CLINIC,49436.23,2601.9068421052634,12,19,19,7
Robert Davis,ONCOLOGY,WESTSIDE CLINIC,40166.49999999999,3089.7307692307686,12,13,13,8
David Jones,PEDIATRICS,CENTRAL HOSPITAL,39315.95,2808.282142857143,13,14,14,9
Sarah Smith,PEDIATRICS,CENTRAL HOSPITAL,37440.91,2202.4064705882356,15,17,17,10


hospital_branch,total_revenue,patients,doctors,appointments,treatments
WESTSIDE CLINIC,160179.30999999994,36,3,54,54
EASTSIDE CLINIC,162031.1,34,3,62,62
CENTRAL HOSPITAL,229039.44000000006,39,4,84,84


year,month,total_revenue,average_bill,total_bills
2023,9,33426.53,3038.7754545454545,11
2023,8,41958.670000000006,2797.244666666667,15
2023,7,39880.19,2492.511875,16
2023,6,56887.81999999999,3160.434444444444,18
2023,3,47304.29000000001,2489.699473684211,19
2023,2,36669.689999999995,2619.263571428571,14
2023,11,52474.979999999996,3086.7635294117645,17
2023,4,64271.539999999986,2570.8615999999993,25
2023,5,48791.04999999999,2567.9499999999994,19
2023,10,43314.149999999994,3093.867857142857,14


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


doctor_name,specialization,hospital_branch,total_revenue,average_bill,total_patients,total_appointments,total_treatments,doctor_rank
Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,82696.47999999998,2851.602758620689,23,29,29,1
Alex Davis,PEDIATRICS,CENTRAL HOSPITAL,69586.1,2899.4208333333336,19,24,24,2
David Taylor,DERMATOLOGY,WESTSIDE CLINIC,66585.39,2663.4156,20,25,25,3
Jane Davis,PEDIATRICS,EASTSIDE CLINIC,59803.45999999999,2847.783809523809,17,21,21,4
Linda Brown,DERMATOLOGY,WESTSIDE CLINIC,53427.42,3339.21375,12,16,16,5
Jane Smith,PEDIATRICS,EASTSIDE CLINIC,52791.40999999999,2399.609545454545,18,22,22,6
Linda Wilson,ONCOLOGY,EASTSIDE CLINIC,49436.23,2601.9068421052634,12,19,19,7
Robert Davis,ONCOLOGY,WESTSIDE CLINIC,40166.49999999999,3089.7307692307686,12,13,13,8
David Jones,PEDIATRICS,CENTRAL HOSPITAL,39315.95,2808.282142857143,13,14,14,9
Sarah Smith,PEDIATRICS,CENTRAL HOSPITAL,37440.91,2202.4064705882356,15,17,17,10


patient_id,First_Name,Last_Name,total_spending,appointments,patient_rank
P012,Laura,Davis,30053.08,10,1
P049,David,Moore,23554.06,7,2
P016,Michael,Taylor,22967.940000000002,7,3
P036,Michael,Wilson,21583.56,7,4
P025,Robert,Wilson,19513.17,5,5
P005,David,Wilson,18609.91,8,6
P035,David,Wilson,18407.420000000002,7,7
P048,Emily,Miller,17082.48,5,8
P010,Michael,Taylor,15929.15,6,9
P017,Jane,Jones,14850.28,4,10


# Appointment Summary

In [0]:
appointment_summary = (
    analytics_df
    .groupBy("status")
    .agg(
        F.countDistinct("appointment_id").alias("total_appointments")
    )
)
appointment_summary.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}appointment_summary")

display(appointment_summary)

status,total_appointments
SCHEDULED,51
CANCELLED,51
NO-SHOW,52
COMPLETED,46


# Patient No-Show KPI

In [0]:
total_appointments = analytics_df.select("appointment_id").distinct().count()

no_show = analytics_df.filter(
    F.upper(F.col("status")) == "NO SHOW"
).select("appointment_id").distinct().count()

no_show_rate = round((no_show / total_appointments) * 100, 2)

kpi_no_show = spark.createDataFrame(
    [("Patient No Show Rate", no_show_rate, "<10%")],
    ["KPI", "Value", "Target"]
)

kpi_no_show.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}kpi_no_show")


# Payment Method Analysis

In [0]:
payment_analysis = (
    analytics_df
    .groupBy("payment_method")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("bill_id").alias("total_bills"),
        F.avg("amount").alias("average_bill")
    )
)

payment_analysis.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}payment_analysis")

display(payment_analysis)

payment_method,total_revenue,total_bills,average_bill
INSURANCE,182160.2799999999,64,2846.2543749999986
CASH,167707.13999999998,61,2749.29737704918
CREDIT CARD,201382.43,75,2685.0990666666667


# Billing Accuracy KPI

In [0]:
billing_accuracy = (
    analytics_df
    .groupBy()
    .agg(
        (
            F.sum(
                F.when(F.col("payment_status") == "PAID", 1).otherwise(0)
            )
            / F.count("bill_id") * 100
        ).alias("billing_accuracy")
    )
)

billing_accuracy.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}kpi_billing_accuracy")

display(billing_accuracy)

billing_accuracy
32.0


# Treatment Analytics

In [0]:

treatment_analysis = (
    analytics_df
    .groupBy("treatment_type")
    .agg(
        F.count("*").alias("total_treatments"),
        F.sum("cost").alias("total_cost"),
        F.avg("cost").alias("average_cost"),
        F.max("cost").alias("maximum_cost"),
        F.min("cost").alias("minimum_cost")
    )
)

treatment_analysis.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}treatment_analysis")

display(treatment_analysis)

treatment_type,total_treatments,total_cost,average_cost,maximum_cost,minimum_cost
X-RAY,41,110653.66999999998,2698.8699999999994,4973.63,806.78
ECG,38,96224.24,2532.2168421052634,4960.65,582.05
MRI,36,116098.16,3224.948888888889,4966.18,662.72
CHEMOTHERAPY,49,128855.67999999998,2629.70775510204,4964.71,534.03
PHYSIOTHERAPY,36,99418.09999999998,2761.613888888888,4846.2,956.39


# Doctor Performance

In [0]:

doctor_performance = (
    analytics_df
    .groupBy(
        "doctor_name",
        "specialization",
        "hospital_branch"
    )
    .agg(
        F.countDistinct("appointment_id").alias("appointments"),
        F.countDistinct("patient_id").alias("patients"),
        F.sum("amount").alias("revenue"),
        F.avg("cost").alias("avg_treatment_cost")
    )
)

doctor_performance.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}doctor_performance")

display(doctor_performance)

doctor_name,specialization,hospital_branch,appointments,patients,revenue,avg_treatment_cost
Alex Davis,PEDIATRICS,CENTRAL HOSPITAL,24,19,69586.1,2899.4208333333336
Jane Smith,PEDIATRICS,EASTSIDE CLINIC,22,18,52791.40999999999,2399.609545454545
Robert Davis,ONCOLOGY,WESTSIDE CLINIC,13,12,40166.49999999999,3089.7307692307686
Sarah Smith,PEDIATRICS,CENTRAL HOSPITAL,17,15,37440.91,2202.4064705882356
Linda Wilson,ONCOLOGY,EASTSIDE CLINIC,19,12,49436.23,2601.9068421052634
Linda Brown,DERMATOLOGY,WESTSIDE CLINIC,16,12,53427.42,3339.21375
David Taylor,DERMATOLOGY,WESTSIDE CLINIC,25,20,66585.39,2663.4156
Jane Davis,PEDIATRICS,EASTSIDE CLINIC,21,17,59803.45999999999,2847.783809523809
Sarah Taylor,DERMATOLOGY,CENTRAL HOSPITAL,29,23,82696.47999999998,2851.602758620689
David Jones,PEDIATRICS,CENTRAL HOSPITAL,14,13,39315.95,2808.282142857143


# Branch Performance

In [0]:
branch_performance = (
    analytics_df
    .groupBy("hospital_branch")
    .agg(
        F.sum("amount").alias("revenue"),
        F.countDistinct("patient_id").alias("patients"),
        F.countDistinct("doctor_name").alias("doctors"),
        F.countDistinct("appointment_id").alias("appointments"),
        F.countDistinct("treatment_id").alias("treatments")
    )
)

branch_performance.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}branch_performance")

display(branch_performance)

hospital_branch,revenue,patients,doctors,appointments,treatments
WESTSIDE CLINIC,160179.30999999994,36,3,54,54
EASTSIDE CLINIC,162031.1,34,3,62,62
CENTRAL HOSPITAL,229039.44000000006,39,4,84,84


#Revenue per Patient

In [0]:
revenue_per_patient = (
    analytics_df
    .groupBy("patient_id", "First_Name", "Last_Name")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

revenue_per_patient.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}revenue_per_patient")

#Revenue per Doctor

In [0]:

revenue_per_doctor = (
    analytics_df
    .groupBy("doctor_name")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

revenue_per_doctor.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}revenue_per_doctor")

# Operating Cost per Patient

In [0]:

operating_cost = (
    analytics_df
    .groupBy("patient_id")
    .agg(
        (
            F.sum("cost") + F.sum("amount")
        ).alias("operating_cost")
    )
)

operating_cost.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}operating_cost_per_patient")

In [0]:
total_patients = analytics_df.select("patient_id").distinct().count()

total_doctors = analytics_df.select("doctor_name").distinct().count()

total_appointments = analytics_df.select("appointment_id").distinct().count()

completed_appointments = analytics_df.filter(
    F.upper(F.col("status")) == "COMPLETED"
).count()

cancelled_appointments = analytics_df.filter(
    F.upper(F.col("status")) == "CANCELLED"
).count()

no_show_appointments = analytics_df.filter(
    F.upper(F.col("status")) == "NO SHOW"
).count()

total_revenue = analytics_df.agg(
    F.sum("amount")
).first()[0]

average_bill = analytics_df.agg(
    F.avg("amount")
).first()[0]

highest_bill = analytics_df.agg(
    F.max("amount")
).first()[0]

lowest_bill = analytics_df.agg(
    F.min("amount")
).first()[0]

average_treatment_cost = analytics_df.agg(
    F.avg("cost")
).first()[0]

total_treatment_cost = analytics_df.agg(
    F.sum("cost")
).first()[0]

paid_bills = analytics_df.filter(
    F.col("payment_status")=="PAID"
).count()

pending_bills = analytics_df.filter(
    F.col("payment_status")=="PENDING"
).count()
     

In [0]:
top_doctor = (
    revenue_by_doctor
    .orderBy(F.desc("total_revenue"))
    .first()["doctor_name"]
)

top_branch = (
    revenue_by_branch
    .orderBy(F.desc("total_revenue"))
    .first()["hospital_branch"]
)

top_treatment = (
    treatment_analysis
    .orderBy(F.desc("total_treatments"))
    .first()["treatment_type"]
)

top_patient = (
    top_patients
    .orderBy(F.desc("total_spending"))
    .first()["patient_id"]
)

most_used_payment = (
    payment_analysis
    .orderBy(F.desc("total_bills"))
    .first()["payment_method"]
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
revenue_per_patient = round(
    total_revenue / total_patients,
    2
)

revenue_per_doctor = round(
    total_revenue / total_doctors,
    2
)

billing_accuracy = round(
    (paid_bills/(paid_bills+pending_bills))*100,
    2
)

no_show_rate = round(
    (no_show_appointments/total_appointments)*100,
    2
)

In [0]:
dashboard = spark.createDataFrame([(

total_patients,

total_doctors,

total_appointments,

completed_appointments,

cancelled_appointments,

no_show_appointments,

round(total_revenue,2),

round(average_bill,2),

round(highest_bill,2),

round(lowest_bill,2),

round(average_treatment_cost,2),

round(total_treatment_cost,2),

top_doctor,

top_branch,

top_treatment,

top_patient,

most_used_payment,

revenue_per_patient,

revenue_per_doctor,

billing_accuracy,

no_show_rate

)],

[
"total_patients",
"total_doctors",
"total_appointments",
"completed_appointments",
"cancelled_appointments",
"no_show_appointments",
"total_revenue",
"average_bill",
"highest_bill",
"lowest_bill",
"average_treatment_cost",
"total_treatment_cost",
"top_doctor",
"top_branch",
"top_treatment",
"top_patient",
"most_used_payment",
"revenue_per_patient",
"revenue_per_doctor",
"billing_accuracy_pct",
"no_show_rate_pct"
])

In [0]:
dashboard.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}executive_dashboard")
     

In [0]:

kpi_summary = spark.createDataFrame([

("Patient No Show Rate", float(no_show_rate), "<10%"),
("Billing Accuracy", float(billing_accuracy), ">98%"),
("Revenue Per Patient", float(revenue_per_patient), "Maximize"),
("Revenue Per Doctor", float(revenue_per_doctor), "Maximize"),
("Total Revenue", float(round(total_revenue,2)), "Increase"),
("Average Bill", float(round(average_bill,2)), "Increase"),
("Average Treatment Cost", float(round(average_treatment_cost,2)), "Monitor"),
("Total Patients", float(total_patients), "Increase"),
("Total Doctors", float(total_doctors), "Optimize"),
("Total Appointments", float(total_appointments), "Increase")

], ["kpi","value","target"])

In [0]:

kpi_summary.write \
.format("delta") \
.mode("overwrite") \
.save(f"{gold_path}kpi_summary")

In [0]:
display(dbutils.fs.ls(gold_path))

path,name,size,modificationTime
abfss://gold@medallionhealthdata03.dfs.core.windows.net/analytics_dataset/,analytics_dataset/,0,1783941444000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/appointment_summary/,appointment_summary/,0,1783941633000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/branch_performance/,branch_performance/,0,1783941770000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/doctor_performance/,doctor_performance/,0,1783941729000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/executive_dashboard/,executive_dashboard/,0,1783941957000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_billing_accuracy/,kpi_billing_accuracy/,0,1783941698000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_no_show/,kpi_no_show/,0,1783941658000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_summary/,kpi_summary/,0,1783941967000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/monthly_revenue/,monthly_revenue/,0,1783941568000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/operating_cost_per_patient/,operating_cost_per_patient/,0,1783941868000


In [0]:
display(
    spark.read
    .format("delta")
    .load(f"{gold_path}executive_dashboard")
)

display(
    spark.read
    .format("delta")
    .load(f"{gold_path}kpi_summary")
)
     

total_patients,total_doctors,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,total_revenue,average_bill,highest_bill,lowest_bill,average_treatment_cost,total_treatment_cost,top_doctor,top_branch,top_treatment,top_patient,most_used_payment,revenue_per_patient,revenue_per_doctor,billing_accuracy_pct,no_show_rate_pct
48,10,200,46,51,0,551249.85,2756.25,4973.63,534.03,2756.25,551249.85,Sarah Taylor,CENTRAL HOSPITAL,CHEMOTHERAPY,P012,CREDIT CARD,11484.37,55124.99,48.12,0.0


kpi,value,target
Patient No Show Rate,0.0,<10%
Billing Accuracy,48.12,>98%
Revenue Per Patient,11484.37,Maximize
Revenue Per Doctor,55124.99,Maximize
Total Revenue,551249.85,Increase
Average Bill,2756.25,Increase
Average Treatment Cost,2756.25,Monitor
Total Patients,48.0,Increase
Total Doctors,10.0,Optimize
Total Appointments,200.0,Increase


In [0]:

display(dbutils.fs.ls(gold_path))

path,name,size,modificationTime
abfss://gold@medallionhealthdata03.dfs.core.windows.net/analytics_dataset/,analytics_dataset/,0,1783941444000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/appointment_summary/,appointment_summary/,0,1783941633000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/branch_performance/,branch_performance/,0,1783941770000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/doctor_performance/,doctor_performance/,0,1783941729000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/executive_dashboard/,executive_dashboard/,0,1783941957000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_billing_accuracy/,kpi_billing_accuracy/,0,1783941698000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_no_show/,kpi_no_show/,0,1783941658000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/kpi_summary/,kpi_summary/,0,1783941967000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/monthly_revenue/,monthly_revenue/,0,1783941568000
abfss://gold@medallionhealthdata03.dfs.core.windows.net/operating_cost_per_patient/,operating_cost_per_patient/,0,1783941868000


In [0]:
display(
    spark.read
    .format("delta")
    .load(f"{gold_path}patients_360")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8315148472061470>, line 1
----> 1 display(
      2     spark.read
      3     .format("delta")
      4     .load(f"{gold_path}patients_360")
      5 )

File /databricks/python_shell/lib/dbruntime/display.py:135, in Display.display(self, input, *args, **kwargs)
    133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
--> 135     if input.isStreaming:
    136         handleStreamingConnectDataFramePy4j(input, self.entry_point, kwargs)
    137     else:

File /usr/lib/python3.12/functools.py:995, in cached_property.__get__(self, instance, owner)
    993 val = cache.get(self.attrname, _NOT_FOUND)
    994 if val is _NOT_FOUND:
--> 995     val = self.func(instance)
    996     try:
    997         cache[self.attrname] = val

File /databricks/spark/python/pyspark/sql/connect/datafr